# HLB Flow — LightGBM training
Synthetic competition data only. Upload the supplied workbook when requested. Training uses April, tuning early May, calibration late May, final test June. Output: `model/` and reports.


In [ ]:
%pip -q install lightgbm==4.6.0 openpyxl scikit-learn


In [ ]:
import pathlib, base64, sys, os
project=pathlib.Path('/content/hlb-flow')
for d in ['training','data','model','reports']: (project/d).mkdir(parents=True,exist_ok=True)
encoded=(
'IiIiUmVwcm9kdWNpYmxlIGNvbXBldGl0aW9uLWxhYmVsIG1vZGVsLiBObyBoaXN0b3JpY2FsIGRlY2lzaW9ucyBlbnRlciBwcmVkaWN0b3JzLiIiIgpmcm9t'
'IHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGFyZ3BhcnNlLCBjb2xsZWN0aW9ucywgZGF0ZXRpbWUgYXMgZHQsIGhhc2hsaWIsIGpzb24sIHBsYXRmb3Jt'
'CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgb3BlbnB5eGwKaW1wb3J0IGxpZ2h0Z2JtIGFzIGxnYgppbXBvcnQgc2tsZWFybgpmcm9tIHNrbGVhcm4ubGlu'
'ZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb24KZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBtYWtlX3BpcGVsaW5lCmZyb20gc2tsZWFy'
'bi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlcgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZSwgYXZlcmFnZV9w'
'cmVjaXNpb25fc2NvcmUsIGJyaWVyX3Njb3JlX2xvc3MsIGxvZ19sb3NzLCBjb25mdXNpb25fbWF0cml4CgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2'
'ZSgpLnBhcmVudHNbMV0KRkVBVFVSRVMgPSBbJ2hhc19mb3JtYWxfY2ljX2hpc3RvcnknLCdtb250aGx5X2luZmxvd192bmQnLCdpbmZsb3dfY3YnLCdjYXNh'
'X2JhbGFuY2Vfdm5kJywnZHRpX3JhdGlvJywnb25fdGltZV9iaWxsX3JhdGlvJywnZV93YWxsZXRfdG9wdXBzX3BtJywncmVndWxhcl93ZWVrbHlfcGF0dGVy'
'bicsJ3JhcGlkX2RyYWluX2ZsYWcnLCdyZXF1ZXN0ZWRfbG9hbl92bmQnLCdsb2FuX3RlbnVyZV9tb250aHMnLCdyZXF1ZXN0ZWRfdG9faW5mbG93JywnY2Fz'
'YV90b19pbmZsb3cnXQpWRVJTSU9OID0gJ2Zsb3ctbGdibS12MScKTElNSVRBVElPTlMgPSBbCiAnU3ludGhldGljIGNvbXBldGl0aW9uIGRhdGE7IG5vdCB2'
'YWxpZGF0ZWQgZm9yIHJlYWwgbGVuZGluZy4nLAogJ2lzX2RlZmF1bHRfOTBkIG1lYW5zIDkwKyBkYXlzIHBhc3QgZHVlIGR1cmluZyBsb2FuIHRlbnVyZSwg'
'bm90IGRlZmF1bHQgd2l0aGluIDkwIGRheXMuJywKICdBbGwgcG9zaXRpdmUgbGFiZWxzIG9jY3VyIGFtb25nIGFwcGxpY2F0aW9ucyBkZWNsaW5lZCBieSBi'
'b3RoIHN1cHBsaWVkIGRlY2lzaW9uIG1ldGhvZHM7IG91dGNvbWUgcHJvdmVuYW5jZSByZXF1aXJlcyBjb25maXJtYXRpb24uJywKICdPdXRjb21lIG1hdHVy'
'aXR5IGRhdGVzIGFuZCByZXBlYXRlZC1wZXJzb24gaWRlbnRpZmllcnMgYXJlIHVuYXZhaWxhYmxlLicsCiAnUHJpbWFyeSBtb2RlbCBleGNsdWRlcyBvcGFx'
'dWUgdW5pZmllZF9jcmVkaXRfc2NvcmUsIGFzc2lnbmVkIGludGVyZXN0IHJhdGUsIElEcywgc2VnbWVudCBhbmQgYWxsIGhpc3RvcmljYWwgZGVjaXNpb25z'
'L291dGNvbWVzLicsCiAnTW9kZWwgb3V0cHV0IGlzIGEgc2ltdWxhdGVkIHJpc2sgZXN0aW1hdGUuIERlbW8gdGhyZXNob2xkcyBhbmQgcHJpY2luZyBhcmUg'
'aWxsdXN0cmF0aXZlLicsCl0KCmRlZiB2ZWN0b3Iocik6CiAgICBkID0gZGljdChyKQogICAgaW5jb21lID0gZmxvYXQoZFsnbW9udGhseV9pbmZsb3dfdm5k'
'J10pCiAgICBpZiBpbmNvbWUgPD0gMDogcmFpc2UgVmFsdWVFcnJvcignSW5jb21lIG11c3QgYmUgcG9zaXRpdmUnKQogICAgZFsncmVxdWVzdGVkX3RvX2lu'
'ZmxvdyddID0gZmxvYXQoZFsncmVxdWVzdGVkX2xvYW5fdm5kJ10pIC8gaW5jb21lCiAgICBkWydjYXNhX3RvX2luZmxvdyddID0gZmxvYXQoZFsnY2FzYV9i'
'YWxhbmNlX3ZuZCddKSAvIGluY29tZQogICAgdmFsdWVzID0gW2Zsb2F0KGRba10pIGZvciBrIGluIEZFQVRVUkVTXQogICAgaWYgbm90IGFsbChucC5pc2Zp'
'bml0ZSh2YWx1ZXMpKTogcmFpc2UgVmFsdWVFcnJvcignTm9uLWZpbml0ZSBmZWF0dXJlcycpCiAgICByZXR1cm4gdmFsdWVzCgpkZWYgbWV0cmljcyh5LCBw'
'KToKICAgIHAgPSBucC5jbGlwKG5wLmFzYXJyYXkocCksIDFlLTksIDEtMWUtOSkKICAgIHJldHVybiB7J24nOmxlbih5KSwncG9zaXRpdmVzJzppbnQoc3Vt'
'KHkpKSwgJ3ByZXZhbGVuY2UnOmZsb2F0KG5wLm1lYW4oeSkpLAogICAgICAncm9jX2F1Yyc6ZmxvYXQocm9jX2F1Y19zY29yZSh5LHApKSBpZiBsZW4oc2V0'
'KHkpKT4xIGVsc2UgTm9uZSwKICAgICAgJ3ByX2F1Yyc6ZmxvYXQoYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeSxwKSksICdicmllcic6ZmxvYXQoYnJpZXJf'
'c2NvcmVfbG9zcyh5LHApKSwKICAgICAgJ2xvZ19sb3NzJzpmbG9hdChsb2dfbG9zcyh5LHAsbGFiZWxzPVswLDFdKSksCiAgICAgICdjb25mdXNpb25fYXRf'
'MDA1Jzpjb25mdXNpb25fbWF0cml4KHkscD49LjA1LGxhYmVscz1bMCwxXSkudG9saXN0KCl9CgpkZWYgbWFpbihzb3VyY2U9Tm9uZSk6CiAgICBzb3VyY2Ug'
'PSBQYXRoKHNvdXJjZSBvciBST09ULydkYXRhL3NvdXJjZS54bHN4JykKICAgIHcgPSBvcGVucHl4bC5sb2FkX3dvcmtib29rKHNvdXJjZSwgcmVhZF9vbmx5'
'PVRydWUsIGRhdGFfb25seT1UcnVlKQogICAgZGVmaW5pdGlvbnMgPSB7cm93WzBdOnJvd1sxXSBmb3Igcm93IGluIHdbJ2RhdGEgZGVmaW5pdGlvbiddLml0'
'ZXJfcm93cyhtaW5fcm93PTIsdmFsdWVzX29ubHk9VHJ1ZSkgaWYgcm93WzBdfQogICAgaXQ9d1snZGF0YXNldCddLml0ZXJfcm93cyh2YWx1ZXNfb25seT1U'
'cnVlKTsgaGVhZGVyPW5leHQoaXQpCiAgICByb3dzPVtdCiAgICBmb3IgdmFscyBpbiBpdDoKICAgICAgICBpZiB2YWxzWzBdIGlzIE5vbmU6IGNvbnRpbnVl'
'CiAgICAgICAgcj1kaWN0KHppcChoZWFkZXIsdmFscykpCiAgICAgICAgZGF0ZT1yWydhcHBsaWNhdGlvbl9kYXRlJ10KICAgICAgICBpZiBpc2luc3RhbmNl'
'KGRhdGUsKGZsb2F0LGludCkpOiBkYXRlPWR0LmRhdGV0aW1lKDE4OTksMTIsMzApK2R0LnRpbWVkZWx0YShkYXlzPWRhdGUpCiAgICAgICAgclsnYXBwbGlj'
'YXRpb25fZGF0ZSddPWRhdGUuc3RyZnRpbWUoJyVZLSVtLSVkJykgaWYgaGFzYXR0cihkYXRlLCdzdHJmdGltZScpIGVsc2Ugc3RyKGRhdGUpWzoxMF0KICAg'
'ICAgICByb3dzLmFwcGVuZChyKQogICAgYXVkaXQ9eydzb3VyY2Vfc2hhMjU2JzpoYXNobGliLnNoYTI1Nihzb3VyY2UucmVhZF9ieXRlcygpKS5oZXhkaWdl'
'c3QoKSwncm93cyc6bGVuKHJvd3MpLCdmaWVsZHMnOmxlbihoZWFkZXIpLAogICAgICAgJ3VuaXF1ZV9hcHBsaWNhdGlvbnMnOmxlbih7clsnYXBwbGljYW50'
'X2lkJ10gZm9yIHIgaW4gcm93c30pLCAncG9zaXRpdmVfbGFiZWxzJzpzdW0oaW50KHJbJ2lzX2RlZmF1bHRfOTBkJ10pIGZvciByIGluIHJvd3MpLAogICAg'
'ICAgJ3Vub2JzZXJ2YWJsZSc6c3VtKG5vdCByWydoYXNfb2JzZXJ2YWJsZV90eCddIGZvciByIGluIHJvd3MpLAogICAgICAgJ3Vuc3VwcG9ydGVkX3RlbnVy'
'ZSc6c3VtKHJbJ2xvYW5fdGVudXJlX21vbnRocyddIG5vdCBpbiBbMyw2LDksMTIsMTgsMjRdIGZvciByIGluIHJvd3MpLAogICAgICAgJ3Bvc2l0aXZlX2xh'
'YmVsc19hbW9uZ19tYW51YWxfYXBwcm92YWxzJzpzdW0oaW50KHJbJ2lzX2RlZmF1bHRfOTBkJ10pIGZvciByIGluIHJvd3MgaWYgclsnbWFudWFsX2FwcHJv'
'dmFsX3N0YXR1cyddKSwKICAgICAgICdwb3NpdGl2ZV9sYWJlbHNfYW1vbmdfc3VwcGxpZWRfc3RwX2FwcHJvdmFscyc6c3VtKGludChyWydpc19kZWZhdWx0'
'XzkwZCddKSBmb3IgciBpbiByb3dzIGlmIHJbJ3N0cF9hcHByb3ZhbF9zdGF0dXMnXSksCiAgICAgICAnZGVmaW5pdGlvbnMnOmRlZmluaXRpb25zLCAnbGlt'
'aXRhdGlvbnMnOkxJTUlUQVRJT05TfQogICAgZWxpZ2libGU9W107IGV4Y2x1ZGVkPWNvbGxlY3Rpb25zLkNvdW50ZXIoKQogICAgZm9yIHIgaW4gcm93czoK'
'ICAgICAgICBpZiBub3QgclsnaGFzX29ic2VydmFibGVfdHgnXTogZXhjbHVkZWRbJ3Vub2JzZXJ2YWJsZSddKz0xOyBjb250aW51ZQogICAgICAgIGlmIHJb'
'J2xvYW5fdGVudXJlX21vbnRocyddIG5vdCBpbiBbMyw2LDksMTIsMTgsMjRdOiBleGNsdWRlZFsndW5zdXBwb3J0ZWRfdGVudXJlJ10rPTE7IGNvbnRpbnVl'
'CiAgICAgICAgdHJ5OiB2ZWN0b3IocikKICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsVHlwZUVycm9yLEtleUVycm9yKTogZXhjbHVkZWRbJ2ludmFsaWRf'
'ZmVhdHVyZXMnXSs9MTsgY29udGludWUKICAgICAgICBlbGlnaWJsZS5hcHBlbmQocikKICAgIGdyb3VwcyA9IHsndHJhaW4nOltdLCAndHVuaW5nJzpbXSwg'
'J2NhbGlicmF0aW9uJzpbXSwgJ3Rlc3QnOltdfQogICAgZm9yIHIgaW4gZWxpZ2libGU6CiAgICAgICAgZD1yWydhcHBsaWNhdGlvbl9kYXRlJ10KICAgICAg'
'ICBrZXk9J3RyYWluJyBpZiBkPCcyMDI2LTA1LTAxJyBlbHNlICd0dW5pbmcnIGlmIGQ8JzIwMjYtMDUtMTYnIGVsc2UgJ2NhbGlicmF0aW9uJyBpZiBkPCcy'
'MDI2LTA2LTAxJyBlbHNlICd0ZXN0JwogICAgICAgIGdyb3Vwc1trZXldLmFwcGVuZChyKQogICAgeHk9e2s6KG5wLmFzYXJyYXkoW3ZlY3RvcihyKSBmb3Ig'
'ciBpbiByc10pLG5wLmFzYXJyYXkoW2ludChyWydpc19kZWZhdWx0XzkwZCddKSBmb3IgciBpbiByc10pKSBmb3IgayxycyBpbiBncm91cHMuaXRlbXMoKX0K'
'ICAgIGZvciBrLCh4LHkpIGluIHh5Lml0ZW1zKCk6CiAgICAgICAgaWYgbGVuKHNldCh5KSk8MjogcmFpc2UgVmFsdWVFcnJvcihmJ3trfTogYm90aCBsYWJl'
'bCBjbGFzc2VzIHJlcXVpcmVkJykKICAgIG1vZGVsPWxnYi5MR0JNQ2xhc3NpZmllcihvYmplY3RpdmU9J2JpbmFyeScsbl9lc3RpbWF0b3JzPTUwMCxsZWFy'
'bmluZ19yYXRlPS4wMyxudW1fbGVhdmVzPTE1LG1heF9kZXB0aD00LAogICAgICAgbWluX2NoaWxkX3NhbXBsZXM9NTAsY29sc2FtcGxlX2J5dHJlZT0uOCxz'
'dWJzYW1wbGU9Ljgsc3Vic2FtcGxlX2ZyZXE9MSxyZWdfYWxwaGE9LjEscmVnX2xhbWJkYT0xLAogICAgICAgcmFuZG9tX3N0YXRlPTQyLG5fam9icz0xLHZl'
'cmJvc2l0eT0tMSkKICAgIG1vZGVsLmZpdCgqeHlbJ3RyYWluJ10sZXZhbF9zZXQ9W3h5Wyd0dW5pbmcnXV0sY2FsbGJhY2tzPVtsZ2IuZWFybHlfc3RvcHBp'
'bmcoNTAsdmVyYm9zZT1GYWxzZSldKQogICAgYmFzZWxpbmU9bWFrZV9waXBlbGluZShTdGFuZGFyZFNjYWxlcigpLExvZ2lzdGljUmVncmVzc2lvbihtYXhf'
'aXRlcj0yMDAwLHJhbmRvbV9zdGF0ZT00MikpCiAgICBiYXNlbGluZS5maXQoKnh5Wyd0cmFpbiddKQogICAgIyBBIHNlcGFyYXRlIHRlbXBvcmFsIHBhcnRp'
'dGlvbiBmaXRzIHNpZ21vaWQgY2FsaWJyYXRpb24uIE5vIEp1bmUgZGF0YSBpcyB1c2VkIGhlcmUuCiAgICBkZWYgcmF3KHgpOiByZXR1cm4gbnAuYXNhcnJh'
'eShtb2RlbC5ib29zdGVyXy5wcmVkaWN0KHgscmF3X3Njb3JlPVRydWUpKQogICAgY2FsaWJyYXRvcj1Mb2dpc3RpY1JlZ3Jlc3Npb24oQz0xLjAscmFuZG9t'
'X3N0YXRlPTQyKS5maXQocmF3KHh5WydjYWxpYnJhdGlvbiddWzBdKS5yZXNoYXBlKC0xLDEpLHh5WydjYWxpYnJhdGlvbiddWzFdKQogICAgYT1mbG9hdChj'
'YWxpYnJhdG9yLmNvZWZfWzAsMF0pOyBiPWZsb2F0KGNhbGlicmF0b3IuaW50ZXJjZXB0X1swXSkKICAgIGRlZiBjYWxpYnJhdGVkKHgpOiByZXR1cm4gMS8o'
'MStucC5leHAoLW5wLmNsaXAoYSpyYXcoeCkrYiwtNzAwLDcwMCkpKQogICAgdGVzdHgsdGVzdHk9eHlbJ3Rlc3QnXQogICAgcmVwb3J0cz17azp7J3Jhdyc6'
'bWV0cmljcyh5LG1vZGVsLnByZWRpY3RfcHJvYmEoeClbOiwxXSksJ2NhbGlicmF0ZWQnOm1ldHJpY3MoeSxjYWxpYnJhdGVkKHgpKX0gZm9yIGssKHgseSkg'
'aW4geHkuaXRlbXMoKX0KICAgIHRlc3RfYmFzZWxpbmVzPXsnY29uc3RhbnQnOm1ldHJpY3ModGVzdHksbnAucmVwZWF0KG5wLm1lYW4oeHlbJ3RyYWluJ11b'
'MV0pLGxlbih0ZXN0eSkpKSwKICAgICAgICdsb2dpc3RpYyc6bWV0cmljcyh0ZXN0eSxiYXNlbGluZS5wcmVkaWN0X3Byb2JhKHRlc3R4KVs6LDFdKX0KICAg'
'IHNlZ21lbnRzPXt9CiAgICBwcmVkPWNhbGlicmF0ZWQodGVzdHgpCiAgICBmb3Igc2VnIGluIHNvcnRlZCh7clsnY3VzdG9tZXJfcHJvZmlsZSddIGZvciBy'
'IGluIGdyb3Vwc1sndGVzdCddfSk6CiAgICAgICAgaXg9bnAuYXJyYXkoW3JbJ2N1c3RvbWVyX3Byb2ZpbGUnXT09c2VnIGZvciByIGluIGdyb3Vwc1sndGVz'
'dCddXSk7IHNlZ21lbnRzW3NlZ109bWV0cmljcyh0ZXN0eVtpeF0scHJlZFtpeF0pCiAgICByZWxpYWJpbGl0eT1bXQogICAgZm9yIGxvLGhpIGluIFsoMCwu'
'MDEpLCguMDEsLjA1KSwoLjA1LC4xMiksKC4xMiwuMyksKC4zLDEuMDAwMDAxKV06CiAgICAgICAgaXg9KHByZWQ+PWxvKSYocHJlZDxoaSkKICAgICAgICBy'
'ZWxpYWJpbGl0eS5hcHBlbmQoeydsb3dlcic6bG8sJ3VwcGVyJzptaW4oaGksMSksJ24nOmludChpeC5zdW0oKSksJ21lYW5fc2NvcmUnOmZsb2F0KHByZWRb'
'aXhdLm1lYW4oKSkgaWYgaXguYW55KCkgZWxzZSBOb25lLCdsYWJlbF9yYXRlJzpmbG9hdCh0ZXN0eVtpeF0ubWVhbigpKSBpZiBpeC5hbnkoKSBlbHNlIE5v'
'bmV9KQogICAgbWFuaWZlc3Q9eyd2ZXJzaW9uJzpWRVJTSU9OLCdmZWF0dXJlcyc6RkVBVFVSRVMsJ2Jlc3RfaXRlcmF0aW9uJzptb2RlbC5iZXN0X2l0ZXJh'
'dGlvbl8sCiAgICAgICAnY2FsaWJyYXRpb24nOnsnbWV0aG9kJzonc2lnbW9pZCcsJ2NvZWZmaWNpZW50JzphLCdpbnRlcmNlcHQnOmIsJ3BhcnRpdGlvbic6'
'JzIwMjYtMDUtMTYgdGhyb3VnaCAyMDI2LTA1LTMxJ30sCiAgICAgICAndGhyZXNob2xkcyc6eydhcHByb3ZlX2JlbG93JzouMDUsJ2RlY2xpbmVfYXRfb3Jf'
'YWJvdmUnOi4xMiwnc3RhdHVzJzonaWxsdXN0cmF0aXZlIGRlbW8gc2V0dGluZ3M7IG5vdCBiYW5rLWFwcHJvdmVkJ30sCiAgICAgICAnb3V0cHV0JzonQ2Fs'
'aWJyYXRlZCBlc3RpbWF0ZSBvZiBzdXBwbGllZCBzeW50aGV0aWMgOTArIERQRCBsYWJlbCBkdXJpbmcgbG9hbiB0ZW51cmUnLAogICAgICAgJ3NvdXJjZV9z'
'aGEyNTYnOmF1ZGl0Wydzb3VyY2Vfc2hhMjU2J10sJ2xpbWl0YXRpb25zJzpMSU1JVEFUSU9OUywKICAgICAgICd2ZXJzaW9ucyc6eydweXRob24nOnBsYXRm'
'b3JtLnB5dGhvbl92ZXJzaW9uKCksJ2xpZ2h0Z2JtJzpsZ2IuX192ZXJzaW9uX18sJ3NrbGVhcm4nOnNrbGVhcm4uX192ZXJzaW9uX199LAogICAgICAgJ3Bh'
'cnRpdGlvbnMnOntrOnsnbic6bGVuKHJzKSwncG9zaXRpdmVzJzpzdW0oaW50KHJbJ2lzX2RlZmF1bHRfOTBkJ10pIGZvciByIGluIHJzKSwnaWRzJzpbclsn'
'YXBwbGljYW50X2lkJ10gZm9yIHIgaW4gcnNdfSBmb3IgayxycyBpbiBncm91cHMuaXRlbXMoKX0sCiAgICAgICAndHJhaW5lZF9hdCc6ZHQuZGF0ZXRpbWUu'
'bm93KGR0LnRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCl9CiAgICAjIFBpY2sgYWN0dWFsIGVsaWdpYmxlIGhvbGRvdXQgcmVjb3JkcyBvbmx5IEFGVEVSIHRo'
'ZSBtb2RlbCBpcyBmcm96ZW4sIGZvciB3YWxrdGhyb3VnaHMsIG5ldmVyIGZvciB0dW5pbmcuCiAgICBwZXJzb25hPVtdCiAgICBjYW5kaWRhdGVzPWdyb3Vw'
'c1sndGVzdCddCiAgICBhbGxwcmVkPWNhbGlicmF0ZWQodGVzdHgpCiAgICBmb3Iga2luZCxzZWdtZW50IGluIFsoJ2dpZycsJ0dpZyAvIHBsYXRmb3JtIHdv'
'cmtlcnMnKSwoJ3NhbGFyaWVkJywnU2FsYXJpZWQsIHdpdGggY3JlZGl0IGhpc3RvcnknKSwoJ21lcmNoYW50JywnT25saW5lIG1lcmNoYW50cyAoTVNNRSkn'
'KSwoJ3RoaW4nLCdTYWxhcmllZCwgbm8gY3JlZGl0IGhpc3RvcnknKV06CiAgICAgICAgbWF0Y2hlcz1bKHIscCkgZm9yIHIscCBpbiB6aXAoY2FuZGlkYXRl'
'cyxhbGxwcmVkKSBpZiByWydjdXN0b21lcl9wcm9maWxlJ109PXNlZ21lbnRdCiAgICAgICAgaWYgbm90IG1hdGNoZXM6IG1hdGNoZXM9bGlzdCh6aXAoY2Fu'
'ZGlkYXRlcyxhbGxwcmVkKSkKICAgICAgICAjIFByZWZlciBsb3cgcmlzayBmb3IgYSBzaG9wcGluZyBkZW1vLiBUaGUgYWN0dWFsIHNjb3JlIG9mIHRoZSBO'
'RVcgb3JkZXIgaXMgYWx3YXlzIHJlY29tcHV0ZWQuCiAgICAgICAgcixfPW1pbihtYXRjaGVzLGtleT1sYW1iZGEgdDp0WzFdKTsgcGVyc29uYS5hcHBlbmQo'
'eydrZXknOmtpbmQsJ25hbWUnOnsnZ2lnJzonTWluaCBOZ3V5ZW4nLCdzYWxhcmllZCc6J0xpbmggVHJhbicsJ21lcmNoYW50JzonQW5oIFBoYW0nLCd0aGlu'
'JzonTWFpIExlJ31ba2luZF0sICdyb3cnOnJ9KQogICAgYXJ0aWZhY3RzPVJPT1QvJ21vZGVsJzsgcmVwb3J0c2Rpcj1ST09ULydyZXBvcnRzJzsgYXJ0aWZh'
'Y3RzLm1rZGlyKGV4aXN0X29rPVRydWUpOyByZXBvcnRzZGlyLm1rZGlyKGV4aXN0X29rPVRydWUpCiAgICBtb2RlbC5ib29zdGVyXy5zYXZlX21vZGVsKHN0'
'cihhcnRpZmFjdHMvJ2xpZ2h0Z2JtLnR4dCcpKQogICAgbWFuaWZlc3RbJ21vZGVsX3NoYTI1NiddPWhhc2hsaWIuc2hhMjU2KChhcnRpZmFjdHMvJ2xpZ2h0'
'Z2JtLnR4dCcpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKICAgIGZvciBwYXRoLG9iaiBpbiBbKGFydGlmYWN0cy8nbWFuaWZlc3QuanNvbicsbWFuaWZl'
'c3QpLChhcnRpZmFjdHMvJ3BlcnNvbmFzLmpzb24nLHBlcnNvbmEpLChyZXBvcnRzZGlyLydkYXRhX2F1ZGl0Lmpzb24nLHsqKmF1ZGl0LCdlbGlnaWJsZSc6'
'bGVuKGVsaWdpYmxlKSwnZXhjbHVkZWQnOmRpY3QoZXhjbHVkZWQpfSksCiAgICAgICAgKHJlcG9ydHNkaXIvJ21vZGVsX3JlcG9ydC5qc29uJyx7J3BhcnRp'
'dGlvbnMnOnJlcG9ydHMsJ3Rlc3RfYmFzZWxpbmVzJzp0ZXN0X2Jhc2VsaW5lcywndGVzdF9zZWdtZW50cyc6c2VnbWVudHMsJ3JlbGlhYmlsaXR5JzpyZWxp'
'YWJpbGl0eSwnbGltaXRhdGlvbnMnOkxJTUlUQVRJT05TfSksCiAgICAgICAgKHJlcG9ydHNkaXIvJ3NjaGVtYS5qc29uJyx7J2ZpZWxkcyc6W3snbmFtZSc6'
'aywnZGVmaW5pdGlvbic6ZGVmaW5pdGlvbnMuZ2V0KGspLCdjb2x1bW4nOmkrMX0gZm9yIGksayBpbiBlbnVtZXJhdGUoaGVhZGVyKV19KV06CiAgICAgICAg'
'cGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMob2JqLGluZGVudD0yLGRlZmF1bHQ9c3RyKSkKICAgIHByaW50KGpzb24uZHVtcHMoeyd2ZXJzaW9uJzpWRVJT'
'SU9OLCdlbGlnaWJsZSc6bGVuKGVsaWdpYmxlKSwndGVzdCc6cmVwb3J0c1sndGVzdCddLCdiYXNlbGluZXMnOnRlc3RfYmFzZWxpbmVzLCdsaW1pdGF0aW9u'
'cyc6TElNSVRBVElPTlN9LGluZGVudD0yKSkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogICAgcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpOyBwLmFk'
'ZF9hcmd1bWVudCgnLS1zb3VyY2UnKTsgYXJncz1wLnBhcnNlX2FyZ3MoKTsgbWFpbihhcmdzLnNvdXJjZSkK'
)
(project/'training/train.py').write_bytes(base64.b64decode(encoded))
(project/'training/__init__.py').touch()
from google.colab import files
uploaded=files.upload()
name=next(iter(uploaded))
if name.lower().endswith('.xlsx'):
 (project/'data/source.xlsx').write_bytes(uploaded[name])
else:
 import csv, io, datetime, openpyxl
 definitions={'applicant_id': 'Unique alphanumeric identifier assigned to each loan application in the inbound queue.', 'application_date': 'Calendar date on which the application was submitted (YYYY-MM-DD), spanning April 1, 2026 to June 30, 2026.', 'customer_profile': 'Applicant demographic and employment classification segment.', 'has_formal_cic_history': 'Binary flag indicating presence of historical records at the national Credit Information Center (1 = Yes, 0 = No).', 'has_observable_tx': 'Binary flag indicating whether pre-application transaction or banking account records are observable for the applicant (1 = Yes, 0 = No).', 'monthly_inflow_vnd': 'Average monthly cash inflow recorded across observable accounts or partner platforms, denominated in VND.', 'inflow_cv': 'Coefficient of variation of monthly cash inflows (ratio of standard deviation to mean inflow), measuring cash-flow dispersion.', 'unified_credit_score': 'Numerical credit rating assigned to the applicant on a standardized scale from 350 to 850.', 'casa_balance_vnd': 'Average 90-day balance maintained in Current and Savings Accounts (CASA) at the bank, denominated in VND.', 'dti_ratio': 'Debt-to-Income ratio, defined as total estimated monthly debt obligations divided by monthly cash inflow.', 'on_time_bill_ratio': 'Ratio of scheduled utility, telecommunication, and recurring invoices settled on or before the due date (0.0 to 1.0).', 'e_wallet_topups_pm': 'Average count of digital wallet top-up transactions completed per month.', 'regular_weekly_pattern': 'Binary flag indicating whether recurring weekly transfer or transaction cycles are detected (1 = Yes, 0 = No).', 'rapid_drain_flag': 'Indicator flag for account depletion or complete fund withdrawal occurring shortly following a top-up (1 = Yes, 0 = No).', 'requested_loan_vnd': 'Financing facility amount requested by the applicant at point of checkout, denominated in VND.', 'approved_loan_vnd': 'Total credit facility amount approved for disbursement in VND (recorded as 0 if not approved).', 'loan_tenure_months': 'Repayment duration of the requested loan facility, in months.', 'interest_rate_pct': 'Annualized nominal interest rate assigned to the financing facility (% per annum).', 'is_default_90d': 'Binary repayment outcome indicating whether the borrower incurred 90 or more days past due during the loan tenure (1 = Yes, 0 = No).', 'manual_approval_status': 'Application underwriting decision under the standard review process (1 = Approved, 0 = Declined).', 'manual_decision_days': 'Turnaround time from application receipt to final decision under the standard review process, in business days.', 'manual_review_cost_vnd': 'Standard operational underwriting and verification cost incurred per application under manual review.', 'is_abandoned': 'Binary flag indicating whether the applicant exited the onboarding process before final decisioning (1 = Abandoned, 0 = Completed).', 'stp_approval_status': 'Underwriting decision generated by the automated Straight-Through Processing framework (1 = Approved, 0 = Declined).', 'stp_decision_minutes': 'Turnaround time from application submission to automated decision delivery under the STP framework, in minutes.', 'stp_review_cost_vnd': 'Estimated system execution and data query cost incurred per application under the automated STP process.'}
 rows=list(csv.DictReader(io.StringIO(uploaded[name].decode('utf-8-sig'))))
 wb=openpyxl.Workbook(); d=wb.active; d.title='data definition';d.append(['Column','Definition'])
 for k,v in definitions.items():d.append([k,v])
 sheet=wb.create_sheet('dataset'); header=list(rows[0]);sheet.append(header)
 for row in rows:
  sheet.append([datetime.datetime.fromisoformat(row[k]) if k=='application_date' else row[k] if k in ['applicant_id','customer_profile'] else float(row[k].replace(',','')) for k in header])
 wb.save(project/'data/source.xlsx')
 (project/'data/training_source.csv').write_bytes(uploaded[name])
print('Prepared source:',name)


In [ ]:
import runpy
os.chdir(project)
sys.path.insert(0,str(project))
sys.argv=['train.py']
_ = runpy.run_path(str(project/'training/train.py'),run_name='__main__')


In [ ]:
import json, platform, datetime, hashlib, zipfile
report=json.loads((project/'reports/model_report.json').read_text())
print(json.dumps(report['partitions']['test'],indent=2))
provenance={'execution_environment':'Google Colab','executed_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),'source_sha256':hashlib.sha256((project/'data/source.xlsx').read_bytes()).hexdigest(),'note':'Colab training on synthetic competition labels; not validated real lending.'}
if (project/'data/training_source.csv').exists(): provenance['input_csv_sha256']=hashlib.sha256((project/'data/training_source.csv').read_bytes()).hexdigest()
(project/'model/colab_provenance.json').write_text(json.dumps(provenance,indent=2))
with zipfile.ZipFile('/content/hlb_flow_colab_model.zip','w',zipfile.ZIP_DEFLATED) as z:
 for folder in ['model','reports']:
  for f in (project/folder).glob('*'):
   if f.is_file():z.write(f,str(f.relative_to(project)))
print('TRAINING COMPLETE: model and evaluation reports exported.')


In [ ]:
files.download('/content/hlb_flow_colab_model.zip')
